In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import datetime
from scipy import stats

In [ ]:
filepath = r"C:\Users\venta\Documents\Grad Plans\portfolio\Quantium\QVI_data.csv"
df = pd.read_csv(filepath, header=0)
df.head()

In [ ]:
df.info() #checking date format
#making yyyymm column
df['DATE'] = pd.to_datetime(df['DATE'])
df['YEARMONTH'] = df['DATE'].dt.strftime('%Y%m').astype(int)
df.head()

In [ ]:
#defining measure calculations for each store and month
#total sales, number of customers, transactions per customer, chips per customer
#and avg price per unit. calc agg of sums and counts first
measures = df.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    totSales = ('TOT_SALES', 'sum'),
    nCustomers = ('LYLTY_CARD_NBR', 'nunique'),
    nChips = ('PROD_QTY', 'sum'),
    nTxn = ('TXN_ID', 'nunique')
).reset_index()
#calc avgs
measures['nTxnPerCust'] = measures['nTxn']/measures['nCustomers']
measures['nChipsPerCust'] = measures['nChips']/measures['nCustomers']
measures['AvgPricePerUnit'] = measures['totSales']/measures['nChips']
print(measures.head())
print(measures.shape)

In [ ]:
#locate stores with full 12 month obs
store_month_counts = measures['STORE_NBR'].value_counts()
stores_FullObs = store_month_counts[store_month_counts == 12].index
#then separate pretrial window
preTrial_measures = measures[(measures['STORE_NBR'].isin(stores_FullObs)) & (measures['YEARMONTH'] <201902)]
preTrial_measures.shape

In [ ]:
#function for correlation calculating for control stores
def calculateCorrelation(inputTable, metricCol, storeComparison):
    results = [] #empty list
    storeNumbers = inputTable['STORE_NBR'].unique() #grabs list of unique store #
    for store in storeNumbers:
        trial_data = inputTable[inputTable['STORE_NBR'] == storeComparison][metricCol].reset_index(drop=True)
        control_data = inputTable[inputTable['STORE_NBR'] == store][metricCol].reset_index(drop=True)
        if len(trial_data) == 7 and len(control_data) == 7:
            correlation = trial_data.corr(control_data)
        else:
            correlation = None #makes sure both have 7 months of data
        results.append({
            'Store1': storeComparison,
            'Store2': store,
            'corr_measure': correlation
        })
    return results

In [ ]:
def calculateMagnitudeDistance(inputTable, metricCol, storeComparison):
    results = []
    storeNumbers = inputTable['STORE_NBR'].unique()
    
    for store in storeNumbers:
        trial_data = inputTable[inputTable['STORE_NBR'] == storeComparison].sort_values('YEARMONTH').reset_index(drop=True)
        control_data = inputTable[inputTable['STORE_NBR'] == store].sort_values('YEARMONTH').reset_index(drop=True)
        if len(trial_data) == 7 and len(control_data) == 7:
            distance = abs(trial_data[metricCol] - control_data[metricCol])
            for i in range(7):
                results.append({
                    'Store1': storeComparison,
                    'Store2': store,
                    'YEARMONTH': trial_data['YEARMONTH'].iloc[i],
                    'measure': distance.iloc[i]
                })
    dist_df = pd.DataFrame(results)

    #standardization
    min_max = dist_df.groupby(['Store1', 'YEARMONTH'])['measure'].agg(['min', 'max']).reset_index()
    dist_df = pd.merge(dist_df, min_max, on=['Store1', 'YEARMONTH'])
    #formula:
    dist_df['magnitudeMeasure'] = 1 - (dist_df['measure'] - dist_df['min']) / (dist_df['max'] - dist_df['min'])


    #averaging the 7 months to get one final score for every store pair
    finalDistTable = dist_df.groupby(['Store1', 'Store2'])['magnitudeMeasure'].mean().reset_index()
    finalDistTable.rename(columns={'magnitudeMeasure': 'mag_measure'}, inplace=True)

    return finalDistTable


In [ ]:
#finding the twin trial store
#correlation function
trial_store = 86
corr_nSales = calculateCorrelation(preTrial_measures, 'totSales', trial_store)
corr_nCustomers = calculateCorrelation(preTrial_measures, 'nCustomers', trial_store)
corr_nSales = pd.DataFrame(corr_nSales)
corr_nCustomers = pd.DataFrame(corr_nCustomers)

#magnitude function
magnitude_nSales = calculateMagnitudeDistance(preTrial_measures, 'totSales', trial_store)
magnitude_nCustomers = calculateMagnitudeDistance(preTrial_measures, 'nCustomers', trial_store)

#checking
print("--- Sales Correlation ---")
print(corr_nSales.head())

print("--- Sales Magnitude ---")
print(magnitude_nSales.head())

In [ ]:
#merging
score_nSales = pd.merge(corr_nSales, magnitude_nSales, on=['Store1', 'Store2'])
score_nSales['scoreControlSales'] = (score_nSales['corr_measure'] * 0.5) + (score_nSales['mag_measure'] * 0.5)
print("--- Score Sales ---")
print(score_nSales.head())

score_nCustomers = pd.merge(corr_nCustomers, magnitude_nCustomers, on=['Store1', 'Store2'])
score_nCustomers['scoreControlCustomers'] = (score_nCustomers['corr_measure'] * 0.5) + (score_nCustomers['mag_measure'] * 0.5)
print("--- Score Customers ---")
print(score_nCustomers.head())
                                     

In [ ]:
#grand merge
score_Control = pd.merge(score_nSales, score_nCustomers, on=['Store1', 'Store2'])
#cleaning column names
score_Control = score_Control.rename(columns={
    'corr_measure_x':'corr_measure_sales',
    'corr_measure_y':'corr_measure_customers',
    'mag_measure_x':'mag_measure_sales',
    'mag_measure_y':'mag_measure_customers'
})
#calculating average
score_Control['finalControlScore'] = (score_Control['scoreControlSales'] * 0.5) + (score_Control['scoreControlCustomers'] * 0.5)

#sorting
score_Control.sort_values('finalControlScore', ascending=False).head()

In [ ]:
#visual verification of sales comparisons
#isolating the two stores
trial_plot = preTrial_measures[preTrial_measures['STORE_NBR'] == 86]
control_plot = preTrial_measures[preTrial_measures['STORE_NBR'] == 155]

#converting yearmonth to string
trial_plot['YEARMONTH'] = trial_plot['YEARMONTH'].astype(str)
control_plot['YEARMONTH'] = control_plot['YEARMONTH'].astype(str)
pretty_dates = [datetime.datetime.strptime(str(x), '%Y%m').strftime('%b-%y') for x in trial_plot['YEARMONTH']]

#plotting
plt.figure(figsize=(10, 5))
plt.plot(pretty_dates, trial_plot['totSales'], label='Trial (86)', marker='o')
plt.plot(pretty_dates, control_plot['totSales'], label='Control (155)', marker='o')

plt.title('Total Sales: Pre-trial Comparison')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.legend()
plt.show()


In [ ]:
#visual verification of customer comparisons

#plotting
plt.figure(figsize=(10, 5))
plt.plot(pretty_dates, trial_plot['nCustomers'], label='Trial (86)', marker='o')
plt.plot(pretty_dates, control_plot['nCustomers'], label='Control (155)', marker='o', linestyle='--')

plt.title('Total Customers: Pre-trial Comparison')
plt.xlabel('Month')
plt.ylabel('Total Customers')
plt.legend()
plt.show()


In [ ]:
#scaling factor
trial_sum = preTrial_measures[preTrial_measures['STORE_NBR'] == trial_store]['totSales'].sum()
control_sum = preTrial_measures[preTrial_measures['STORE_NBR'] == 155]['totSales'].sum()

#ratio
scaling_factor = trial_sum/control_sum
print(f"Scaling Factor: {scaling_factor}")

#applying scaling factor
#grab every month of control store
controlSales = measures[measures['STORE_NBR'] == 155].copy()
#create scaled column
controlSales['controlSales'] = controlSales['totSales'] * scaling_factor
controlSales[['YEARMONTH', 'totSales', 'controlSales']].head()

In [ ]:
#grabbing trial store 86 data
trialSales = measures[measures['STORE_NBR'] == trial_store][['YEARMONTH', 'totSales']]
trialSales = trialSales.rename(columns={'totSales':'trialSales'})
#merge
percentageDiff_sales = pd.merge(trialSales, controlSales[['YEARMONTH', 'controlSales']], on='YEARMONTH')

#calculate
percentageDiff_sales['percentageDiff'] = (percentageDiff_sales['trialSales'] - percentageDiff_sales['controlSales']) / percentageDiff_sales['controlSales']

percentageDiff_sales.head()

In [ ]:
stdDev_sales = percentageDiff_sales[percentageDiff_sales['YEARMONTH'] < 201902]['percentageDiff'].std()
#t value where null is u=0
percentageDiff_sales['tValue'] = percentageDiff_sales['percentageDiff'] / stdDev_sales
#crit val w 95th percentile and 7 deg of freedom (8 months in pretrial period)
critical_value = stats.t.ppf(0.95, df=7)
print(f"Critical Value: {critical_value}")

In [ ]:
#comparing crit val
trial_months = [201903, 201904, 201905]
trial_results = percentageDiff_sales[percentageDiff_sales['YEARMONTH'].isin(trial_months)]
trial_results['Significant'] = abs(trial_results['tValue']) > critical_value
print(trial_results[['YEARMONTH', 'percentageDiff', 'tValue', 'Significant']])

In [ ]:
#--------interpretations---------
#the experiment increased sales by ~26.8% in March, 
#which is a statistically significant increase at the 95% confidence level, 
#but in April and May, the difference between trial and control store sales increased by 
#~3.3% and decreased by ~1.5% respectively,
#and these differences are not significant at the same confidence level


In [ ]:
#plotting 95% confidence
#calculate bounds
percentageDiff_sales['Control_Upper_95'] = percentageDiff_sales['controlSales'] * (1 + (stdDev_sales * critical_value))
percentageDiff_sales['Control_Lower_5'] = percentageDiff_sales['controlSales'] * (1 - (stdDev_sales * critical_value))

#filter report window jan 2019 -- may 2019
report_df = percentageDiff_sales[percentageDiff_sales['YEARMONTH'] >= 201901]
pretty_dates_report = [datetime.datetime.strptime(str(x), '%Y%m').strftime('%b-%y') for x in report_df['YEARMONTH']]

#plot
plt.figure(figsize=(12, 6))
plt.plot(pretty_dates_report, report_df['trialSales'], label='Trial (86)', marker='o', color='red', linewidth=2)
plt.plot(pretty_dates_report, report_df['controlSales'], label='Control (155)', marker='o', color='blue', linestyle='--')

#adding 95% region
plt.fill_between(pretty_dates_report, report_df['Control_Lower_5'], report_df['Control_Upper_95'],
                 color='gray', alpha=0.2, label='95% Confidence Interval')
plt.title('Trial Period Impact: Store 86 vs Store 155')
plt.ylabel('Total Sales')
plt.legend()
plt.show()

In [ ]:
#------customers store 86
#scaling factor
trial_sum_cust = preTrial_measures[preTrial_measures['STORE_NBR'] == trial_store]['nCustomers'].sum()
control_sum_cust = preTrial_measures[preTrial_measures['STORE_NBR'] == 155]['nCustomers'].sum()

#ratio
scaling_factor_cust = trial_sum_cust/control_sum_cust
print(f"Scaling Factor: {scaling_factor_cust}")

#applying scaling factor
#grab every month of control store
controlCustomers = measures[measures['STORE_NBR'] == 155].copy()
#create scaled column
controlCustomers['controlCustomers'] = controlCustomers['nCustomers'] * scaling_factor_cust
controlCustomers[['YEARMONTH', 'nCustomers', 'controlCustomers']].head()

In [ ]:
#percentageDiff customers
#grabbing trial store 86 data
trialCustomers = measures[measures['STORE_NBR'] == trial_store][['YEARMONTH', 'nCustomers']]
trialCustomers = trialCustomers.rename(columns={'nCustomers':'trialCustomers'})
#merge
percentageDiff_cust = pd.merge(trialCustomers, controlCustomers[['YEARMONTH', 'controlCustomers']], on='YEARMONTH')

#calculate
percentageDiff_cust['percentageDiff'] = (percentageDiff_cust['trialCustomers'] - percentageDiff_cust['controlCustomers']) / percentageDiff_cust['controlCustomers']

percentageDiff_cust.head()

In [ ]:
stdDev_cust = percentageDiff_cust[percentageDiff_cust['YEARMONTH'] < 201902]['percentageDiff'].std()
#t value where null is u=0
percentageDiff_cust['tValue'] = percentageDiff_cust['percentageDiff'] / stdDev_cust

In [ ]:
#comparing crit val customers
trial_results_cust = percentageDiff_cust[percentageDiff_cust['YEARMONTH'].isin(trial_months)]
trial_results_cust['Significant'] = abs(trial_results_cust['tValue']) > critical_value
print(trial_results_cust[['YEARMONTH', 'percentageDiff', 'tValue', 'Significant']])

In [ ]:
#--------interpretations---------
#the experiment increased customer foot traffic by ~18.3% in March 
#which is a statistically significant increase at the 95% confidence level, 
#but in April and May, the difference between trial and control store foot traffic increased by 
#~6.1% and decreased by ~2.3% respectively, and these are not 
#significant differences at the same confidence level

In [ ]:
#plotting 95% confidence
#calculate bounds
percentageDiff_cust['Control_Upper_95'] = percentageDiff_cust['controlCustomers'] * (1 + (stdDev_cust * critical_value))
percentageDiff_cust['Control_Lower_5'] = percentageDiff_cust['controlCustomers'] * (1 - (stdDev_cust * critical_value))

#filter report window jan 2019 -- may 2019
report_cust_df = percentageDiff_cust[percentageDiff_cust['YEARMONTH'] >= 201901]
pretty_dates_report_cust = [datetime.datetime.strptime(str(x), '%Y%m').strftime('%b-%y') for x in report_cust_df['YEARMONTH']]

#plot
plt.figure(figsize=(12, 6))
plt.plot(pretty_dates_report_cust, report_cust_df['trialCustomers'], label='Trial (86)', marker='o', color='red', linewidth=2)
plt.plot(pretty_dates_report_cust, report_cust_df['controlCustomers'], label='Control (155)', marker='o', color='blue', linestyle='--')

#adding 95% region
plt.fill_between(pretty_dates_report_cust, report_cust_df['Control_Lower_5'], report_cust_df['Control_Upper_95'],
                 color='gray', alpha=0.2, label='95% Confidence Interval')
plt.title('Trial Period Impact: Store 86 vs Store 155')
plt.ylabel('Total Customers')
plt.legend()
plt.show()